In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [2]:
from torchvision.datasets import CIFAR10
from torchvision.transforms import transforms

In [3]:
#Creating a Transformer

transform = transforms.Compose([
    transforms.ToTensor(),     # This will help to convert image into Tensordata

    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))          #Setting Standard Deviation and Mean value
])

In [4]:
#Creating a Dataset

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [5]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
#creating a DataLoader
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset, batch_size=64)

# Building CNN

In [19]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        #Multi Convolutional Pooling Layer
        self.conv_layers = nn.Sequential(

            #Layer 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),     #input = 3, op = (n-f+2p/s) = 32
            nn.ReLU(),
            nn.MaxPool2d(2,2),      #Kernal_size = 2, stride = 2

            #Layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),     #input = 32, op = 64
            nn.ReLU(),
            nn.MaxPool2d(2,2),      #Kernal_size = 2, stride = 2

            #Layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),     #input = 64, op = 128
            nn.ReLU(),
            nn.MaxPool2d(2,2),      #Kernal_size = 2, stride = 2
        )

        #CNN Fully Connected Layer
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),                    #input = 4*4*18 Flatten, op=256 (Randomly selected value)
            nn.ReLU(),

            nn.Linear(256, 10)                          #10 = 10 Categories
        )


    def forward(self, x):
            x = self.conv_layers(x)
            x = x.view(x.size(0), -1)           #flatenning
            x = self.fc_layers(x)

            return x


In [20]:
model = CNN()

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN


In [22]:
#Optimizer and Loss Function

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(train_loader)}")

epoch=1/10 & loss=1.400520186899873
epoch=2/10 & loss=0.974617348531323
epoch=3/10 & loss=0.7780135619975722
epoch=4/10 & loss=0.6520404816436036
epoch=5/10 & loss=0.5529602636080568
epoch=6/10 & loss=0.45666040184781376
epoch=7/10 & loss=0.3731672110040779
epoch=8/10 & loss=0.30044261729130356
epoch=9/10 & loss=0.22981090256301187
epoch=10/10 & loss=0.18100841473216367


# Model Evaluation

In [ ]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs,1)


        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

    print(f"accuracy : {correct_labels / total_labels * 100}")

accuracy : 75.55
